# Clients Table Cleaning

**Purpose:** Stores one record per client account commissioning VFX work.

**Expected grain:** One row per client.

**Primary key:** `client_id`

**Important rules:**
- `client_id` must be complete, unique, and follow `CL-###`.
- Client names must be trimmed and use consistent display casing.
- Region must be North America, Europe, or Asia-Pacific.
- Contract tier must be A, B, or C.
- Revision tendency must be a positive decimal.
- Review SLA must be a positive whole number.
- Active status must be Y or N.
- Blank notes are valid and should be stored as null.

In [217]:
# Connect to this notebook's private project snapshot
from pathlib import Path
import os
import duckdb

project_root = Path(os.environ["INSIGHT_PROJECT_ROOT"])
database_path = Path(
    os.environ["INSIGHT_NOTEBOOK_DATABASE"]
)
connection = duckdb.connect(str(database_path))
for folder_name in ('data', 'raw', 'clean', 'public'):
    connection.execute(
        f'CREATE SCHEMA IF NOT EXISTS "{folder_name}"'
    )
connection.execute(
    "SET search_path = 'raw,clean,public,data,main'"
)
try:
    connection.execute("LOAD inflector")
    inflector_available = True
except Exception:
    inflector_available = False
project_tables = connection.execute(
    """
    SELECT
        table_schema AS folder_name,
        table_name,
        table_schema || '.' || table_name AS sql_reference
    FROM information_schema.tables
    WHERE table_schema NOT IN (
        'main',
        'temp',
        'information_schema',
        'pg_catalog'
    )
      AND table_name NOT LIKE '_insight_%'
    ORDER BY table_schema, table_name
    """
).fetchall()
if project_tables:
    print('Project tables:')
    for folder_name, table_name, sql_reference in project_tables:
        print(f'  {sql_reference}')
else:
    print('No project tables are currently available.')

Project tables:
  clean.raw_artists_cleaned
  clean.raw_clients_cleaned
  raw.raw_artists
  raw.raw_clients
  raw.raw_projects
  raw.raw_reviews
  raw.raw_shots
  raw.raw_time_entries
  staging.artists

In [218]:
DESCRIBE raw_clients;

SELECT
    COUNT(*) AS total_rows,
    COUNT(client_id) AS populated_client_ids,
    COUNT(DISTINCT client_id) AS distinct_client_ids,
    COUNT(*) - COUNT(DISTINCT client_id) AS duplicate_rows
FROM raw_clients;

total_rows,populated_client_ids,distinct_client_ids,duplicate_rows
14,14,12,2


In [219]:
SELECT
    client_id,
    COUNT(*) AS row_count
FROM raw_clients
GROUP BY client_id
HAVING COUNT(*) > 1
ORDER BY client_id;

SELECT *
FROM raw_clients
WHERE client_id IN ('CL-007', 'CL-012')
ORDER BY client_id;

client_id,client_name,client_type,region,contract_tier,revision_tendency_index,default_review_sla_hours,active_flag,account_manager,notes
CL-007,Northstar Episodic,Television Studio,North America,B,1.2,20,Y,Nico Martinez,NULL
CL-007,Northstar Episodic,Television Studio,North America,B,1.2,20,Y,Nico Martinez,NULL
CL-012,COPPERFIELD CREATIVE,Agency,Europe,C,0.85,12,Active,Mateo Davis,NULL
CL-012,COPPERFIELD CREATIVE,Agency,Europe,C,0.85,12,Active,Mateo Davis,NULL


## Duplicate decision

`CL-007` and `CL-012` each appear twice as exact duplicate export
records. One canonical copy of each client will be retained. These are
duplicate rows, not separate client accounts.

In [220]:
SELECT
    COUNT(*) FILTER (
        WHERE client_id IS NULL
        OR TRIM(client_id) = ''
    ) AS missing_client_id,

    COUNT(*) FILTER (
        WHERE client_name IS NULL
        OR TRIM(client_name) = ''
    ) AS missing_client_name,

    COUNT(*) FILTER (
        WHERE client_type IS NULL
        OR TRIM(client_type) = ''
    ) AS missing_client_type,

    COUNT(*) FILTER (
        WHERE region IS NULL
        OR TRIM(region) = ''
    ) AS missing_region,

    COUNT(*) FILTER (
        WHERE contract_tier IS NULL
        OR TRIM(contract_tier) = ''
    ) AS missing_contract_tier,

    COUNT(*) FILTER (
        WHERE revision_tendency_index IS NULL
    ) AS missing_revision_index,

    COUNT(*) FILTER (
        WHERE default_review_sla_hours IS NULL
    ) AS missing_review_sla,

    COUNT(*) FILTER (
        WHERE active_flag IS NULL
        OR TRIM(active_flag) = ''
    ) AS missing_active_flag,

    COUNT(*) FILTER (
        WHERE account_manager IS NULL
        OR TRIM(account_manager) = ''
    ) AS missing_account_manager,

    COUNT(*) FILTER (
        WHERE notes IS NULL
        OR TRIM(notes) = ''
    ) AS blank_notes
FROM raw_clients;

missing_client_id,missing_client_name,missing_client_type,missing_region,missing_contract_tier,missing_revision_index,missing_review_sla,missing_active_flag,missing_account_manager,blank_notes
0,0,0,0,0,0,0,0,0,14


In [221]:
SELECT *
FROM raw_clients
WHERE client_id <> TRIM(client_id)
   OR client_name <> TRIM(client_name)
   OR client_type <> TRIM(client_type)
   OR region <> TRIM(region)
   OR contract_tier <> TRIM(contract_tier)
   OR active_flag <> TRIM(active_flag)
   OR account_manager <> TRIM(account_manager);
   
SELECT
    client_id,
    client_name,
    TRIM(client_name) AS trimmed_client_name
FROM raw_clients
WHERE client_name <> TRIM(client_name)
   OR client_name = UPPER(client_name)
ORDER BY client_id;

client_id,client_name,trimmed_client_name
CL-003,SILVER ARC TELEVISION,SILVER ARC TELEVISION
CL-012,COPPERFIELD CREATIVE,COPPERFIELD CREATIVE
CL-012,COPPERFIELD CREATIVE,COPPERFIELD CREATIVE


In [222]:
SELECT
    client_type,
    COUNT(*) AS row_count
FROM raw_clients
GROUP BY client_type
ORDER BY client_type;

client_type,row_count
Agency,3
Feature Studio,3
Game Cinematics,1
Independent Studio,1
Marketing Vendor,1
Streaming Platform,2
Television Studio,3


In [223]:
SELECT
    region,
    COUNT(*) AS row_count
FROM raw_clients
GROUP BY region
ORDER BY region;

region,row_count
Asia-Pacific,2
Europe,5
N. America,1
North America,6


`N. America` is an alternate label and must be standardized to `North America`. It should not remain as a separate reporting region.

In [224]:
SELECT
    contract_tier,
    COUNT(*) AS row_count
FROM raw_clients
GROUP BY contract_tier
ORDER BY contract_tier;

contract_tier,row_count
A,5
B,5
C,4


In [225]:
SELECT
    active_flag,
    COUNT(*) AS row_count
FROM raw_clients
GROUP BY active_flag
ORDER BY active_flag;

active_flag,row_count
Active,2
TRUE,1
Y,10
Yes,1


All should become Y. The cleaned field should permit only Y and N.

In [226]:
SELECT
    client_id,
    client_name,
    revision_tendency_index
FROM raw_clients
WHERE TRY_CAST(revision_tendency_index AS DOUBLE) IS NULL
   OR TRY_CAST(revision_tendency_index AS DOUBLE) <= 0;

client_id,client_name,revision_tendency_index


In [227]:
SELECT
    client_id,
    client_name,
    default_review_sla_hours
FROM raw_clients
WHERE TRY_CAST(default_review_sla_hours AS INTEGER) IS NULL
   OR TRY_CAST(default_review_sla_hours AS INTEGER) <= 0
   OR TRY_CAST(default_review_sla_hours AS DOUBLE)
      <> TRY_CAST(default_review_sla_hours AS INTEGER);

client_id,client_name,default_review_sla_hours


In [228]:
CREATE SCHEMA IF NOT EXISTS clean;

CREATE OR REPLACE TABLE clean.raw_clients_cleaned AS

WITH standardized AS (
    SELECT
        UPPER(TRIM(client_id)) AS client_id,

        CASE UPPER(TRIM(client_name))
            WHEN 'COPPERFIELD CREATIVE'
                THEN 'Copperfield Creative'
            WHEN 'SILVER ARC TELEVISION'
                THEN 'Silver Arc Television'
            ELSE TRIM(client_name)
        END AS client_name,

        TRIM(client_type) AS client_type,

        CASE UPPER(TRIM(region))
            WHEN 'N. AMERICA'
                THEN 'North America'
            WHEN 'NORTH AMERICA'
                THEN 'North America'
            WHEN 'EUROPE'
                THEN 'Europe'
            WHEN 'ASIA-PACIFIC'
                THEN 'Asia-Pacific'
            ELSE TRIM(region)
        END AS region,

        UPPER(TRIM(contract_tier)) AS contract_tier,

        TRY_CAST(
            revision_tendency_index AS DOUBLE
        ) AS revision_tendency_index,

        TRY_CAST(
            default_review_sla_hours AS INTEGER
        ) AS default_review_sla_hours,

        CASE UPPER(TRIM(active_flag))
            WHEN 'Y' THEN 'Y'
            WHEN 'YES' THEN 'Y'
            WHEN 'TRUE' THEN 'Y'
            WHEN 'ACTIVE' THEN 'Y'
            WHEN 'N' THEN 'N'
            WHEN 'NO' THEN 'N'
            WHEN 'FALSE' THEN 'N'
            WHEN 'INACTIVE' THEN 'N'
            ELSE NULL
        END AS active_flag,

        TRIM(account_manager) AS account_manager,

        NULLIF(TRIM(notes), '') AS notes

    FROM raw_clients
)

SELECT DISTINCT *
FROM standardized;

Count
12


In [229]:
SELECT
    COUNT(*) AS cleaned_rows,
    COUNT(client_id) AS populated_client_ids,
    COUNT(DISTINCT client_id) AS distinct_client_ids,
    COUNT(*) - COUNT(DISTINCT client_id) AS duplicate_client_ids
FROM clean.raw_clients_cleaned;

cleaned_rows,populated_client_ids,distinct_client_ids,duplicate_client_ids
12,12,12,0


In [230]:
SELECT DISTINCT
    p.project_id,
    p.client_id AS raw_client_id,
    UPPER(TRIM(p.client_id)) AS standardized_client_id
FROM raw.raw_projects AS p
LEFT JOIN clean.raw_clients_cleaned AS c
    ON UPPER(TRIM(p.client_id)) = c.client_id
WHERE c.client_id IS NULL
ORDER BY p.project_id;

project_id,raw_client_id,standardized_client_id
PRJ-1011,CL-999,CL-999


In [231]:
SELECT *
FROM clean.raw_clients_cleaned
ORDER BY client_id;

client_id,client_name,client_type,region,contract_tier,revision_tendency_index,default_review_sla_hours,active_flag,account_manager,notes
CL-001,Blue Lantern Pictures,Feature Studio,North America,A,1.15,24,Y,Cameron Lee,NULL
CL-002,Signal Peak Streaming,Streaming Platform,North America,A,1.35,18,Y,Blake King,NULL
CL-003,Silver Arc Television,Television Studio,Europe,B,1.05,24,Y,Avery Davis,NULL
CL-004,Kiteframe Advertising,Agency,North America,B,0.9,12,Y,Skyler Davis,NULL
CL-005,Orchid Gate Media,Feature Studio,Asia-Pacific,A,1.25,24,Y,Aisha King,NULL
CL-006,Cinderline Entertainment,Independent Studio,Europe,C,1.1,36,Y,Rowan Davis,NULL
CL-007,Northstar Episodic,Television Studio,North America,B,1.2,20,Y,Nico Martinez,NULL
CL-008,Monument Trailer House,Marketing Vendor,North America,C,0.8,8,Y,Parker Walker,NULL
CL-009,Sunbird Interactive,Game Cinematics,Asia-Pacific,B,1.05,24,Y,Sofia Nguyen,NULL
CL-010,Marble Harbor Films,Feature Studio,Europe,A,1.3,24,Y,Casey King,NULL


## Cleaning decisions and remaining exceptions

The Clients table was cleaned by trimming and standardizing client
identifiers, organization names, category labels, regions, contract tiers,
active-status values, and account-manager names according to the completed
Data Dictionary. Exact duplicate records for CL-007 and CL-012 were removed,
reducing the table from 14 export rows to 12 unique client records. N. America
was standardized to North America, affirmative active-status variants were
mapped to Y, valid numeric revision-tendency and SLA values were preserved,
and blank optional notes were converted to null without imputation. The clean
client table contains a complete and unique primary key. The remaining
client-related exception is the project record referencing CL-999; because no
matching client exists, that issue will be resolved or quarantined during the
Projects cleaning stage rather than inventing a client record.